In [ ]:
from tqdm import tqdm
import zipfile

import os
import shutil
import numpy as np
import pandas as pd
from ultralytics import YOLO
from sklearn.model_selection import train_test_split
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

In [ ]:
def get_image_size(img_path):
    with Image.open(img_path) as img:
        return img.size

def yolo_bbox(row, img_w, img_h, n_classes):
    class_id = int(row['class_id'])
    if class_id >= n_classes:
        return None

    x_min = max(0, min(row['x_min'], img_w))
    y_min = max(0, min(row['y_min'], img_h))
    x_max = max(0, min(row['x_max'], img_w))
    y_max = max(0, min(row['y_max'], img_h))

    if x_max <= x_min or y_max <= y_min:
        return None

    x_center = max(0, min((x_min + x_max) / 2 / img_w, 1))
    y_center = max(0, min((y_min + y_max) / 2 / img_h, 1))
    width = max(0, min((x_max - x_min) / img_w, 1))
    height = max(0, min((y_max - y_min) / img_h, 1))

    return f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"

def generate_yolo_dataset(df, image_ids, img_dir, labels_dir, img_sizes_cache, n_classes):
    """Generate .txt files for specified image_ids"""
    os.makedirs(labels_dir, exist_ok=True)
    for img_id in image_ids:
        img_path = os.path.join(img_dir, f"{img_id}.jpg")
        if img_id not in img_sizes_cache:
            img_sizes_cache[img_id] = get_image_size(img_path)
        img_w, img_h = img_sizes_cache[img_id]

        mask = df['image_id'] == img_id
        txt_path = os.path.join(labels_dir, f"{img_id}.txt")
        with open(txt_path, 'w') as f:
            if not mask.any():
                continue
            for _, row in df[mask].iterrows():
                line = yolo_bbox(row, img_w, img_h, n_classes)
                if line is not None:
                    f.write(line + '\n')

def create_yaml(train_path, val_path, nc=14, names=None):
 
    if names is None:
        names = [f"class_{i}" for i in range(nc)]
    yaml_content = f"""
                    path: .
                    train: {train_path}
                    val: {val_path}
                    nc: {nc}
                    names: {names}
                    """
    yaml_path = "data/dataset.yaml"
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)
    return yaml_path

## Prepare data
you need to download images from https://www.kaggle.com/datasets/sonnlp/vindr-cxr-images
and also train.csv and sample_submission.csv files from https://www.kaggle.com/competitions/vinbigdata-chest-xray-abnormalities-detection/data

If you have enough memory on your disk, you can download the same images from https://www.kaggle.com/competitions/vinbigdata-chest-xray-abnormalities-detection/data. It costs ~200 gb and it is in a better quality 

In [ ]:
df = pd.read_csv("data/VinDr-CXR/images/full.csv")  # замените на ваш файл
n_classes = 14
df_clean = df[df['class_id'] < n_classes].copy()
print(f"📊 All rows: {len(df_clean)}, unique images: {df_clean['image_id'].nunique()}")

In [ ]:
img_dir = "data/VinDr-CXR/images/train"

# Split on train/val. We will use almost all the images for training
image_ids = df_clean['image_id'].unique()
train_ids, val_ids = train_test_split(image_ids, test_size=0.001, random_state=42)
print(f"Train: {len(train_ids)}, Val: {len(val_ids)}")

In [ ]:
dataset_dir = "data/yolo_dataset"
os.makedirs(f"{dataset_dir}/images/train", exist_ok=True)
os.makedirs(f"{dataset_dir}/images/val", exist_ok=True)
os.makedirs(f"{dataset_dir}/labels/train", exist_ok=True)
os.makedirs(f"{dataset_dir}/labels/val", exist_ok=True)

for img_id in tqdm(train_ids, total=len(train_ids)):
    src = os.path.join(img_dir, f"{img_id}.jpg")
    dst = os.path.join(dataset_dir, "images/train", f"{img_id}.jpg")
    if not os.path.exists(dst) and os.path.exists(src):
        shutil.copy2(src, dst)  # копируем с сохранением метаданных

for img_id in tqdm(val_ids, total=len(val_ids)):
    src = os.path.join(img_dir, f"{img_id}.jpg")
    dst = os.path.join(dataset_dir, "images/val", f"{img_id}.jpg")
    if not os.path.exists(dst) and os.path.exists(src):
        shutil.copy2(src, dst)


In [ ]:
img_sizes_cache = {}
generate_yolo_dataset(
    df_clean, train_ids, img_dir,
    f"{dataset_dir}/labels/train",
    img_sizes_cache, n_classes
)
generate_yolo_dataset(
    df_clean, val_ids, img_dir,
    f"{dataset_dir}/labels/val",
    img_sizes_cache, n_classes
)

# Create YAML
yaml_path = create_yaml(
    os.path.abspath(f"{dataset_dir}/images/train"),
    os.path.abspath(f"{dataset_dir}/images/val"),
    nc=n_classes
)

## Train the model

In [ ]:
model = YOLO('data/yolov8m.pt')

results = model.train(
    data=yaml_path,
    epochs=1,              
    imgsz=640,
    batch=16,               
    device="cpu",                
    project='yolo_training',
    name='vindr_improved',
    exist_ok=True,
    patience=20,             
    lr0=0.01,                
    augment=False,           
    seed=42,
    verbose=True
)

## Prepare predictions

In [ ]:
MODEL_PATH = "data/runs/detect/yolo_training/vindr_improved/weights/best.pt"
TEST_IMAGES_DIR = "data/vindr-cxr-images/images/test"
CONFIDENCE_THRESHOLD = 0.3

model = YOLO(MODEL_PATH)
print(f"✅ Model is loaded: {MODEL_PATH}")

# Upload sample_submission.csv to get list of all image_id
sample_submission = pd.read_csv("data/VinDr-CXR/images/sample_submission.csv")
test_image_ids = sample_submission['image_id'].unique()
print(f"📊 All images for predictions: {len(test_image_ids)}")

submission_rows = []

for image_id in tqdm(test_image_ids, desc="Processing images"):
    image_path = os.path.join(TEST_IMAGES_DIR, f"{image_id}.jpg")
    
    if not os.path.exists(image_path):
        continue
    
    results = model(image_path, verbose=False)
    
    detections = []
    
    if results[0].boxes is not None and len(results[0].boxes) > 0:
        boxes = results[0].boxes
        for box in boxes:
            conf = float(box.conf[0])
            if conf < CONFIDENCE_THRESHOLD:
                continue
            
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            class_id = int(box.cls[0])
            
            # Format: class_id confidence x_min y_min x_max y_max
            detections.append(f"{class_id} {conf:.6f} {x1:.6f} {y1:.6f} {x2:.6f} {y2:.6f}")
    
    if not detections:
        detections.append("14 1.000000 0.000000 0.000000 1.000000 1.000000")
    
    submission_rows.append({
        'image_id': image_id,
        'prediction_string': ' '.join(detections)
    })

## Save submission df, that you can submit on kaggle competition - https://www.kaggle.com/datasets/sonnlp/vindr-cxr-images

In [ ]:
submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv('data/submission.csv', index=False)
print(f"✅ submission.csv saved ({len(submission_df)} rows)")

print(submission_df.head(3))

In [ ]:
def zip_folder(folder_path, zip_name):
    if not os.path.exists(folder_path):
        print(f"⚠️ Dir {folder_path} not found")
        return
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                arcname = os.path.relpath(file_path, start=os.path.dirname(folder_path))
                zipf.write(file_path, arcname)
    size_mb = os.path.getsize(zip_name) / (1024 * 1024)
    print(f"✅ Архив {zip_name} создан ({size_mb:.2f} MB)")

# Архивируем папку с обучением
zip_folder("dataruns/detect/yolo_training", "data/yolo_training.zip")